In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


In [ ]:
PASTA_REAIS = 'modelo/reais'   
PASTA_FAKES = 'modelo/fake'    
EXTENSOES   = ['*.jpg', '*.jpeg', '*.png', '*.webp']
TAMANHO     = 256               

def listar_imagens(pasta):
    arquivos = []
    for ext in EXTENSOES:
        arquivos.extend(Path(pasta).glob(ext))
    return sorted(arquivos)

imagens_reais = listar_imagens(PASTA_REAIS)
imagens_fake  = listar_imagens(PASTA_FAKES)

print(f'Reais encontradas: {len(imagens_reais)}')
print(f'Fake  encontradas: {len(imagens_fake)}')

In [ ]:
def carregar_luminancia(caminho_imagem):
    img = cv2.imread(str(caminho_imagem))
    img  = cv2.resize(img, (TAMANHO, TAMANHO))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return gray.astype(np.float32)

In [ ]:
def calcular_gradientes(luminancia):
    # Verificação de pixeis vizinhos (Gx: horizontal, Gy: vertical)
    lum_uint8 = luminancia.astype(np.uint8)
    Gx = cv2.Sobel(lum_uint8, cv2.CV_64F, 1, 0, ksize=3)
    Gy = cv2.Sobel(lum_uint8, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(Gx**2 + Gy**2)
    return Gx, Gy, magnitude

In [ ]:
def extrair_features(caminho_imagem):
    lum = carregar_luminancia(caminho_imagem)
    Gx, Gy, magnitude = calcular_gradientes(lum)
    features = np.concatenate([ # Vetor com as features
        Gx.flatten(),       
        Gy.flatten()        
    ])                      
    return features

X, y = [], []
erros = 0

print('Processando imagens REAIS...')
for img in imagens_reais:
    try:
        X.append(extrair_features(img))
        y.append(0)   # 0 = real
    except Exception as e:
        erros += 1
        print(f'  Erro: {img.name} — {e}')

print('Processando imagens FAKE...')
for img in imagens_fake:
    try:
        X.append(extrair_features(img))
        y.append(1)   # 1 = fake
    except Exception as e:
        erros += 1
        print(f'  Erro: {img.name} — {e}')

X = np.array(X)
y = np.array(y)

print(f'\nDataset montado: {X.shape}')
print(f'  {(y==0).sum()} reais | {(y==1).sum()} fake | {erros} erros')

In [ ]:
import joblib
import os

os.makedirs('modelo', exist_ok=True)

# Salvar scaler e pca
joblib.dump(scaler, 'modelo/scaler.pkl')
joblib.dump(pca,    'modelo/pca.pkl')

print('Modelo salvo!')
print('  modelo/scaler.pkl')
print('  modelo/pca.pkl')

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca   = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'PC1 explica: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'PC2 explica: {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total: {sum(pca.explained_variance_ratio_)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))


ax1  = axes[0]
cores = ['#3B82F6' if label == 0 else '#EF4444' for label in y]
ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=cores, alpha=0.7, s=80,
            edgecolors='white', lw=0.5)

real_patch = mpatches.Patch(color='#3B82F6', label='Real')
fake_patch = mpatches.Patch(color='#EF4444', label='Fake (IA)')
ax1.legend(handles=[real_patch, fake_patch])
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax1.set_title('Separação Real vs. Fake no Espaço PCA')
ax1.grid(alpha=0.2)


pca_full       = PCA().fit(X_scaled)
variancia_acum = np.cumsum(pca_full.explained_variance_ratio_)

ax2 = axes[1]
ax2.plot(variancia_acum, color='#10B981', lw=2)
ax2.axhline(y=0.95, color='red', linestyle='--', alpha=0.7, label='95% variância')
ax2.set_xlabel('Número de Componentes')
ax2.set_ylabel('Variância Explicada Acumulada')
ax2.set_title('Scree Plot')
ax2.legend()
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('resultado_pca.png', dpi=150)
plt.show()
print('Gráfico salvo como resultado_pca.png')